# Drug Dataset - Tuned Random Forest Classifier

Notebook nay fine-tune Random Forest cho bo du lieu drug. Ngoai RandomizedSearchCV cho hyperparameter, notebook con tune nguong du doan bang out-of-fold probabilities tren train set de cai thien recall/F1 cua class 1.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_val_predict, cross_validate
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42
RUN_RANDOM_SEARCH = True


def find_project_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "data").exists():
            return path
    raise FileNotFoundError("Cannot find project root containing data/ directory")


ROOT = find_project_root(Path.cwd().resolve())
DATA_DIR = ROOT / "data" / "drug"

train_df = pd.read_csv(DATA_DIR / "DIA_trainingset_RDKit_descriptors.csv")
test_df = pd.read_csv(DATA_DIR / "DIA_testset_RDKit_descriptors.csv")

train_df.shape, test_df.shape

In [ ]:
display(train_df.head())
print("Train label distribution (%):")
display(train_df["Label"].value_counts(normalize=True).mul(100).round(2))
print("Test label distribution (%):")
display(test_df["Label"].value_counts(normalize=True).mul(100).round(2))

In [ ]:
def split_xy(df: pd.DataFrame):
    y = df["Label"].astype(int)
    X = df.drop(columns=["Label", "SMILES"], errors="ignore")
    X = X.select_dtypes(include=[np.number]).replace([np.inf, -np.inf], np.nan)
    return X, y


X_train, y_train = split_xy(train_df)
X_test, y_test = split_xy(test_df)
X_test = X_test.reindex(columns=X_train.columns)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("Missing train cells:", int(X_train.isna().sum().sum()))
print("Missing test cells:", int(X_test.isna().sum().sum()))

In [ ]:
base_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("variance", VarianceThreshold(threshold=0.0)),
        ("model", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)),
    ]
)

param_distributions = {
    "model__n_estimators": [400, 600, 800, 1000],
    "model__max_features": ["sqrt", "log2", 0.25, 0.5],
    "model__max_depth": [None, 6, 10, 14, 18],
    "model__min_samples_split": [2, 4, 8, 12],
    "model__min_samples_leaf": [1, 2, 4, 6],
    "model__class_weight": ["balanced", "balanced_subsample"],
    "model__bootstrap": [True],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

if RUN_RANDOM_SEARCH:
    search = RandomizedSearchCV(
        base_pipeline,
        param_distributions=param_distributions,
        n_iter=45,
        scoring="roc_auc",
        cv=cv,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        refit=True,
        verbose=1,
    )
    search.fit(X_train, y_train)
    rf_model = search.best_estimator_
    print("Best CV ROC-AUC:", round(search.best_score_, 4))
    print("Best params:", search.best_params_)
else:
    rf_model = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("variance", VarianceThreshold(threshold=0.0)),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=800,
                    max_features=0.5,
                    max_depth=None,
                    min_samples_split=2,
                    min_samples_leaf=1,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                ),
            ),
        ]
    )
    rf_model.fit(X_train, y_train)

In [ ]:
scoring = ["roc_auc", "balanced_accuracy", "f1", "accuracy"]
cv_result = cross_validate(
    rf_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
    return_train_score=False,
)

pd.DataFrame(cv_result).filter(like="test_").agg(["mean", "std"]).T.round(4)

In [ ]:
def threshold_table(y_true, y_score):
    rows = []
    for threshold in np.round(np.arange(0.05, 0.96, 0.01), 2):
        pred = (y_score >= threshold).astype(int)
        rows.append(
            {
                "threshold": threshold,
                "balanced_accuracy": balanced_accuracy_score(y_true, pred),
                "f1": f1_score(y_true, pred, zero_division=0),
                "accuracy": accuracy_score(y_true, pred),
            }
        )
    return pd.DataFrame(rows).sort_values(["balanced_accuracy", "f1"], ascending=False).reset_index(drop=True)


cv_proba = cross_val_predict(rf_model, X_train, y_train, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]
threshold_candidates = threshold_table(y_train, cv_proba)
best_threshold = float(threshold_candidates.loc[0, "threshold"])

print("Best threshold from train CV:", best_threshold)
display(threshold_candidates.head(10).round(4))

In [ ]:
def evaluate_model(model, X, y, threshold=0.5):
    y_proba = model.predict_proba(X)[:, 1]
    y_pred = (y_proba >= threshold).astype(int)
    metrics = {
        "threshold": threshold,
        "accuracy": accuracy_score(y, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y, y_pred),
        "f1": f1_score(y, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y, y_proba),
    }
    display(pd.Series(metrics).round(4).to_frame("test_score"))
    display(pd.DataFrame(confusion_matrix(y, y_pred), index=["true_0", "true_1"], columns=["pred_0", "pred_1"]))
    print(classification_report(y, y_pred, digits=4))


print("Default threshold = 0.50")
evaluate_model(rf_model, X_test, y_test, threshold=0.5)

print("Tuned threshold")
evaluate_model(rf_model, X_test, y_test, threshold=best_threshold)

In [ ]:
selector = rf_model.named_steps["variance"]
model = rf_model.named_steps["model"]
selected_features = X_train.columns[selector.get_support()]

importance_df = pd.DataFrame(
    {
        "feature": selected_features,
        "importance": model.feature_importances_,
    }
).sort_values("importance", ascending=False)

display(importance_df.head(30))

out_path = Path.cwd() / "drug_random_forest_feature_importance.csv"
importance_df.to_csv(out_path, index=False)
print("Saved:", out_path)